SETUP - Aprendizaje Profundo con YOLO
==========================================================

Este notebook prepara el entorno de trabajo descargando el conjunto de datos
COCO 2017 (subconjunto de validación) y generando un subset de 100 imágenes
para realizar las pruebas de detección de objetos con los modelos YOLO.

El conjunto COCO (Common Objects in Context) es ampliamente utilizado en
tareas de detección y segmentación debido a su diversidad de clases y
la calidad de sus anotaciones.

### Importaciones y Rutas/URLs

In [ ]:
import os
import requests
import zipfile
from pathlib import Path
from tqdm import tqdm

# --- CONFIGURACIÓN ---
# Directorios base donde se guardarán los datos
BASE_DIR = Path('datasets/coco')
IMG_DIR = BASE_DIR / 'val2017'
ANN_DIR = BASE_DIR / 'annotations'
VAL_JSON = ANN_DIR / 'instances_val2017.json'

# URLs oficiales proporcionadas en la documentación
URL_IMAGES = "http://images.cocodataset.org/zips/val2017.zip"
URL_ANNOTATIONS = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"

### Funciones auxiliares
Aquí implementamos las funciones necesarias para gestionar los datos de forma automatizada:
* `download_file`: Gestiona la descarga de archivos grandes, incorporando una barra de progreso visual para monitorear el estado.
* `unzip_file`: Se encarga de descomprimir los archivos .zip en las rutas especificadas.
* `create_subset`: Implementa la lógica crítica del proyecto. Ordena alfabéticamente las imágenes descargadas y conserva únicamente las primeras 100, eliminando el resto para cumplir con los requisitos de la práctica y agilizar el procesamiento.

In [ ]:
def download_file(url, dest_path):
    """Descarga un archivo desde una URL con barra de progreso."""
    if dest_path.exists():
        print(f"El archivo {dest_path.name} ya existe. Saltando descarga.")
        return

    print(f"Descargando {url}...")
    response = requests.get(url, stream=True)
    total_size = int(response.headers.get('content-length', 0))
    
    # Crear directorio padre si no existe
    dest_path.parent.mkdir(parents=True, exist_ok=True)

    with open(dest_path, 'wb') as file, tqdm(
        desc=dest_path.name,
        total=total_size,
        unit='iB',
        unit_scale=True,
        unit_divisor=1024,
    ) as bar:
        for data in response.iter_content(chunk_size=1024):
            size = file.write(data)
            bar.update(size)


def unzip_file(zip_path, extract_to):
    """Descomprime un archivo zip en el directorio destino."""
    print(f"Descomprimiendo {zip_path.name} en {extract_to}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    print("Descompresión completada.")


def create_subset(img_dir, limit=100):
    """
    Mantiene solo las primeras 'limit' imágenes en el directorio 
    y elimina el resto para crear el subconjunto.
    """
    print(f"\nGenerando subconjunto de las primeras {limit} imágenes...")
    
    # Obtener lista de imágenes ordenadas para ser deterministas
    images = sorted([f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.jpeg', '.png'))])
    
    if len(images) <= limit:
        print(f"La carpeta ya tiene {len(images)} imágenes (menor o igual al límite). No se borra nada.")
        return

    # Separar las que se quedan y las que se van
    files_to_keep = set(images[:limit])
    files_to_remove = images[limit:]
    
    print(f"Se conservarán {len(files_to_keep)} imágenes. Eliminando {len(files_to_remove)} archivos restantes...")
    
    for filename in files_to_remove:
        file_path = img_dir / filename
        os.remove(file_path)
        
    print("¡Subconjunto generado exitosamente!")

### Ejecución
Este bloque orquesta todo el proceso de preparación del entorno:
1.  **Estructura de Directorios**: Verifica y crea las carpetas base.
2.  **Descarga y Extracción**: Obtiene los archivos oficiales de COCO 2017 (imágenes de validación y anotaciones) y los descomprime.
3.  **Reducción del Dataset**: Llama a `create_subset` para eliminar el exceso de imágenes, dejando solo las 100 necesarias.
4.  **Limpieza**: Elimina los archivos .zip descargados para liberar espacio en disco.

Al finalizar, tendremos la carpeta `datasets/coco/val2017` lista para ser usada en los notebooks de tutorial y evaluación.

In [ ]:
# Verificar si el conjunto de datos ya está listo (100 imágenes)
dataset_ready = False
if IMG_DIR.exists() and VAL_JSON.exists():
    num_images = len([f for f in os.listdir(IMG_DIR) if f.endswith(('.jpg', '.png', '.jpeg'))])
    if num_images == 100:
        print(f"El dataset ya está configurado correctamente con {num_images} imágenes.")
        print("No es necesario descargar nada. Setup finalizado.")
        dataset_ready = True

if not dataset_ready:
    print("Dataset no encontrado o incompleto. Iniciando descarga...")
    # Crear directorios base
    if not BASE_DIR.exists():
        BASE_DIR.mkdir(parents=True)

    # Descargar y descomprimir Anotaciones (solo nos interesa el JSON de validación)
    # Verificar si ya existe
    if not VAL_JSON.exists():
        zip_ann_path = BASE_DIR / "annotations_trainval2017.zip"
        download_file(URL_ANNOTATIONS, zip_ann_path)
        with zipfile.ZipFile(zip_ann_path, 'r') as zip_ref:
            # Filtramos solo los archivos que contienen 'val2017' en su nombre
            # Esto evita extraer los archivos gigantes de 'train2017'
            files_to_extract = [f for f in zip_ref.namelist() if 'val2017' in f]
            
            for file in files_to_extract:
                print(f"  -> Extrayendo: {file}")
                zip_ref.extract(file, BASE_DIR)
        # Limpiar archivo ZIP de anotaciones
        if zip_ann_path.exists():
            os.remove(zip_ann_path)
    else:
        print(f"Anotaciones de validación ya presentes. Saltando descarga.")

    # Descargar y descomprimir Imágenes
    # Solo descargamos si no existe la carpeta descomprimida o no son 100 imágenes
    if not IMG_DIR.exists() or len(os.listdir(IMG_DIR)) != 100:
        zip_img_path = BASE_DIR / "val2017.zip"
        download_file(URL_IMAGES, zip_img_path)
        unzip_file(zip_img_path, BASE_DIR)
    
        # Generar el subconjunto (100 primeras imágenes)
        if IMG_DIR.exists():
            create_subset(IMG_DIR, limit=100)
        # Limpiar archivo ZIP de imágenes
        if zip_img_path.exists():
            os.remove(zip_img_path)
    else:
        print(f"El directorio de imágenes {IMG_DIR} ya existe con 100 imágenes. Saltando descarga.")

    print("\n--- Setup finalizado exitosamente ---")